In [2]:
import numpy as np 
import os
from model_evaluation_helpers import compute_ranking_metrics
from scipy.special import expit

In [3]:
model_output_dir = os.path.join(os.path.expanduser("~"), "Downloads", "ClassifierLayer")
outputs = {
    "Large": np.load(os.path.join(model_output_dir, "y_pred_large.npy")), 
    "Medium": np.load(os.path.join(model_output_dir, "y_pred_medium.npy")), 
    "Small": np.load(os.path.join(model_output_dir, "y_pred_small.npy"))
}
y_true = np.load(os.path.join(model_output_dir, "y_true.npy"))

In [4]:
evaluated_models = {key: compute_ranking_metrics(y_true, value) for key, value in outputs.items()}

In [5]:
from tabulate import tabulate

labels = ["STTC", "HYP", "MI", "CD", "AF"]
model_keys = ["Large", "Medium", "Small"]

def youdens_j(model, idx):
    return model['per_label_recall'][idx] + model['per_label_specificity'][idx] - 1

results = [
    ["Macro_F1"] + [evaluated_models[k]['macro_f1'] for k in model_keys],
    ["Macro_AUROC"] + [evaluated_models[k]['macro_auroc'] for k in model_keys],
    ["Micro_AUROC"] + [evaluated_models[k]['micro_auroc'] for k in model_keys],
    ["Hyp Sensitivity"] + [evaluated_models[k]['per_label_recall'][1] for k in model_keys],
    ["Hyp Specificity"] + [evaluated_models[k]['per_label_specificity'][1] for k in model_keys],
    [""] * (len(model_keys) + 1),
    *[[f"{lbl} Youden's J"] + [youdens_j(evaluated_models[k], i) for k in model_keys] for i, lbl in enumerate(labels)],
    [""] * (len(model_keys) + 1),
    *[[f"{lbl}_F1"] + [evaluated_models[k]['per_label_f1'][i] for k in model_keys] for i, lbl in enumerate(labels)],
]

print(tabulate(
    results, 
    headers=["Metric", "Large Classifier", "Medium Classifier", "Small Classifier"], 
    tablefmt="github"
))

| Metric          | Large Classifier   | Medium Classifier   | Small Classifier   |
|-----------------|--------------------|---------------------|--------------------|
| Macro_F1        | 0.7704536886950308 | 0.7676934546347295  | 0.7746470110875234 |
| Macro_AUROC     | 0.9374329292156384 | 0.9384969487588848  | 0.9371936726933224 |
| Micro_AUROC     | 0.9442108518804615 | 0.9446118078018637  | 0.9431145898710955 |
| Hyp Sensitivity | 0.6432432432432432 | 0.7135135135135136  | 0.5972972972972973 |
| Hyp Specificity | 0.9532674772036475 | 0.9319908814589666  | 0.9677051671732523 |
|                 |                    |                     |                    |
| STTC Youden's J | 0.6958060371540427 | 0.685070967546896   | 0.7015505539852169 |
| HYP Youden's J  | 0.5965107204468907 | 0.6455043949724801  | 0.5650024644705496 |
| MI Youden's J   | 0.6845999718785152 | 0.7055164979377577  | 0.7065558211473566 |
| CD Youden's J   | 0.7341024821203197 | 0.7194473706352547  | 0.73520572149